# Build hourly online features with missing PM as NaN

This notebook rebuilds the online streaming CSV from the full weather grid and raw Pulse Eco measurements.

Goal:
- keep one hourly row per sensor/timestamp from the weather grid
- attach observed PM10/PM2.5 where Pulse Eco has data
- leave missing PM10/PM2.5 as blank CSV cells, which pandas reads back as NaN
- avoid interpolation, ffill, bfill, or median filling

In [13]:
from pathlib import Path

import pandas as pd

In [14]:
# Paths
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "feature_engineering":
    PROJECT_ROOT = PROJECT_ROOT.parent

SOURCE_WEATHER_CSV = PROJECT_ROOT / "data" / "streaming" / "bitola_sensor_weather_features_online.csv"
PULSE_DIR = PROJECT_ROOT / "feature_engineering" / "pulse_data"
OUTPUT_CSV = PROJECT_ROOT / "data" / "streaming" / "bitola_sensor_weather_features_online_hourly_nan.csv"

# Keep the same sensor-quality rule as the observed-only CSV.
# Set to None if you want to keep every sensor, including sensors with 100% missing PM values.
MAX_MISSING_PERCENT = 50.0

SOURCE_WEATHER_CSV, PULSE_DIR, OUTPUT_CSV

(PosixPath('/mnt/c/Users/RazorVision/Desktop/project-vrnmp/data/streaming/bitola_sensor_weather_features_online.csv'),
 PosixPath('/mnt/c/Users/RazorVision/Desktop/project-vrnmp/feature_engineering/pulse_data'),
 PosixPath('/mnt/c/Users/RazorVision/Desktop/project-vrnmp/data/streaming/bitola_sensor_weather_features_online_hourly_nan.csv'))

## 1. Load the weather grid

The original online CSV already has the hourly weather rows we need. We drop the old PM columns because those were filled/imputed before. Then we join raw Pulse PM values again.

In [15]:
weather = pd.read_csv(SOURCE_WEATHER_CSV)
weather = weather.drop(columns=["pm10", "pm25"], errors="ignore")

weather["timestamp"] = pd.to_datetime(weather["timestamp"], utc=True, errors="coerce").dt.floor("h")
weather["sensorId"] = weather["sensorId"].astype(str)

weather = (
    weather
    .dropna(subset=["timestamp", "sensorId"])
    .sort_values(["sensorId", "timestamp"])
    .drop_duplicates(subset=["sensorId", "timestamp"], keep="last")
    .reset_index(drop=True)
)

print(f"Weather grid rows: {len(weather):,}")
print(f"Sensors in weather grid: {weather['sensorId'].nunique():,}")
print(f"Date range: {weather['timestamp'].min()} -> {weather['timestamp'].max()}")
weather.head()

Weather grid rows: 48,026
Sensors in weather grid: 22
Date range: 2025-12-01 00:00:00+00:00 -> 2026-03-01 22:00:00+00:00


,timestamp,sensorId,lat,lon,temperature_2m,relative_humidity_2m,wind_speed_10m,wind_direction_10m,surface_pressure
0,2025-12-01 00:00:00+00:00,16836a55-7140-43e2-9a63-56fac5cba714,41.023537,21.330127,1.9505,92.41695,4.293669,303.02386,943.6906
1,2025-12-01 01:00:00+00:00,16836a55-7140-43e2-9a63-56fac5cba714,41.023537,21.330127,2.0005,92.41994,3.818376,315.00010,944.0742
2,2025-12-01 02:00:00+00:00,16836a55-7140-43e2-9a63-56fac5cba714,41.023537,21.330127,1.6505,93.06822,3.893995,326.30990,944.1686
3,2025-12-01 03:00:00+00:00,16836a55-7140-43e2-9a63-56fac5cba714,41.023537,21.330127,1.4005,93.72927,4.104631,322.12494,944.4741
4,2025-12-01 04:00:00+00:00,16836a55-7140-43e2-9a63-56fac5cba714,41.023537,21.330127,1.3005,94.06355,3.563818,315.00010,944.8185


## 2. Load raw Pulse PM measurements

Pulse timestamps are converted to UTC and floored to the hour so they align with the weather grid. If multiple readings land in the same sensor/hour/type, we use the mean.

In [16]:
pulse_files = sorted(PULSE_DIR.rglob("*.csv"))
print(f"Pulse files found: {len(pulse_files):,}")

pulse_parts = []
for file_path in pulse_files:
    part = pd.read_csv(file_path)
    part["source_file"] = str(file_path.relative_to(PROJECT_ROOT))
    pulse_parts.append(part)

pulse = pd.concat(pulse_parts, ignore_index=True)
pulse = pulse[pulse["type"].isin(["pm10", "pm25"])].copy()

pulse["timestamp"] = pd.to_datetime(pulse["timestamp"], utc=True, errors="coerce").dt.floor("h")
pulse["sensorId"] = pulse["sensorId"].astype(str)
pulse["value"] = pd.to_numeric(pulse["value"], errors="coerce")

pulse = pulse.dropna(subset=["timestamp", "sensorId", "type", "value"])

hourly_pm = (
    pulse
    .groupby(["sensorId", "timestamp", "type"], as_index=False)["value"]
    .mean()
    .pivot(index=["sensorId", "timestamp"], columns="type", values="value")
    .reset_index()
    .rename_axis(columns=None)
)

for column in ["pm10", "pm25"]:
    if column not in hourly_pm.columns:
        hourly_pm[column] = pd.NA

hourly_pm = hourly_pm[["sensorId", "timestamp", "pm10", "pm25"]]

print(f"Hourly PM rows: {len(hourly_pm):,}")
print(f"Sensors with any PM data: {hourly_pm['sensorId'].nunique():,}")
hourly_pm.head()

Pulse files found: 286


/tmp/ipykernel_8802/3921039424.py:10: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  pulse = pd.concat(pulse_parts, ignore_index=True)


Hourly PM rows: 21,585
Sensors with any PM data: 12


,sensorId,timestamp,pm10,pm25
0,16836a55-7140-43e2-9a63-56fac5cba714,2025-12-01 00:00:00+00:00,34.75,17.50
1,16836a55-7140-43e2-9a63-56fac5cba714,2025-12-01 01:00:00+00:00,18.25,10.00
2,16836a55-7140-43e2-9a63-56fac5cba714,2025-12-01 02:00:00+00:00,15.00,8.75
3,16836a55-7140-43e2-9a63-56fac5cba714,2025-12-01 03:00:00+00:00,14.75,7.25
4,16836a55-7140-43e2-9a63-56fac5cba714,2025-12-01 04:00:00+00:00,19.00,8.75


## 3. Join PM onto the hourly weather grid

This keeps all weather-grid rows. Missing PM stays missing.

In [17]:
merged = weather.merge(
    hourly_pm,
    on=["sensorId", "timestamp"],
    how="left",
)

missing_by_sensor = (
    merged
    .groupby("sensorId")[["pm10", "pm25"]]
    .apply(lambda frame: frame.isna().mean() * 100)
    .round(2)
    .sort_values("pm10", ascending=False)
)

has_pm_data = (
    merged
    .groupby("sensorId")[["pm10", "pm25"]]
    .apply(lambda frame: frame.notna().any())
)

print("Sensors with any PM data:")
print(has_pm_data.sum())

missing_by_sensor

Sensors with any PM data:
pm10    12
pm25    12
dtype: int64


,pm10,pm25
sensorId,,
ece1058a-ecab-4736-872f-790145aaadfe,100.00,100.00
24039f11-a4fc-4b2d-8bc0-6fd36059f117,100.00,100.00
30dab8a6-ff63-43ce-9a3b-99f1f3f7054d,100.00,100.00
e20e9778-a020-4b86-932a-b7ab6a713a00,100.00,100.00
692c454c-a1ad-41fa-b3ca-aa1cb7d55d30,100.00,100.00
d851c0b9-990e-41db-9c53-529f88524cf9,100.00,100.00
a17013e7-8d1d-4b0d-8e2f-e0881dbca3ac,100.00,100.00
a9a2083f-f086-4fae-bdae-355b391f436b,100.00,100.00
be427cee-4c3a-4aa2-a1ce-9795a74533be,100.00,100.00


## 4. Optionally drop sensors with too much missing PM data

By default this keeps only sensors where both PM10 and PM2.5 missingness is <= 50%.

Unlike the previous observed-only CSV, this does not drop individual missing rows. It keeps the hourly rows and leaves PM as NaN.

In [18]:
if MAX_MISSING_PERCENT is None:
    kept_sensors = missing_by_sensor.index
else:
    kept_sensors = missing_by_sensor[
        (missing_by_sensor["pm10"] <= MAX_MISSING_PERCENT)
        & (missing_by_sensor["pm25"] <= MAX_MISSING_PERCENT)
    ].index

hourly_nan = (
    merged[merged["sensorId"].isin(kept_sensors)]
    .sort_values(["timestamp", "sensorId"])
    .reset_index(drop=True)
)

print(f"Kept sensors: {len(kept_sensors):,} / {missing_by_sensor.shape[0]:,}")
print(f"Output rows: {len(hourly_nan):,}")
print("Missing PM percent in output:")
print((hourly_nan[["pm10", "pm25"]].isna().mean() * 100).round(2))

hourly_nan.head()

Kept sensors: 10 / 22
Output rows: 21,830
Missing PM percent in output:
pm10    6.79
pm25    6.78
dtype: float64


,timestamp,sensorId,lat,lon,temperature_2m,relative_humidity_2m,wind_speed_10m,wind_direction_10m,surface_pressure,pm10,pm25
0,2025-12-01 00:00:00+00:00,16836a55-7140-43e2-9a63-56fac5cba714,41.023537,21.330127,1.9505,92.41695,4.293669,303.02386,943.69060,34.75,17.50
1,2025-12-01 00:00:00+00:00,2002,41.030221,21.336733,1.9635,92.41773,4.293669,303.02386,943.92490,115.00,103.00
2,2025-12-01 00:00:00+00:00,23b735ef-a996-4a7f-9998-2aa7e78827b0,41.050417,21.346658,2.0675,92.42392,4.293669,303.02386,945.80180,93.75,41.25
3,2025-12-01 00:00:00+00:00,40f081a6-4095-43f7-bffb-64e2af8c026e,41.040130,21.340139,2.0850,100.00000,6.681856,265.36462,939.46530,56.75,34.75
4,2025-12-01 00:00:00+00:00,7b316592-8036-41e2-b8dc-b06b6a9afd54,41.038285,21.327864,2.0330,100.00000,6.681856,265.36462,938.53284,93.50,56.50


In [19]:
# filter to show only nan values for pm10 and pm25
hourly_nan[hourly_nan[["pm10", "pm25"]].isna().any(axis=1)]


,timestamp,sensorId,lat,lon,temperature_2m,relative_humidity_2m,wind_speed_10m,wind_direction_10m,surface_pressure,pm10,pm25
5,2025-12-01 00:00:00+00:00,874ff9c6-786d-45fc-a90e-48c7ffe03417,41.027636,21.311089,2.098000,100.000000,6.681856,265.364620,939.69867,NaN,NaN
15,2025-12-01 01:00:00+00:00,874ff9c6-786d-45fc-a90e-48c7ffe03417,41.027636,21.311089,2.448000,100.000000,6.504951,255.579200,940.07050,NaN,NaN
25,2025-12-01 02:00:00+00:00,874ff9c6-786d-45fc-a90e-48c7ffe03417,41.027636,21.311089,2.448000,100.000000,5.297018,260.217650,940.34735,NaN,NaN
35,2025-12-01 03:00:00+00:00,874ff9c6-786d-45fc-a90e-48c7ffe03417,41.027636,21.311089,1.998000,100.000000,4.978554,257.471200,940.50160,NaN,NaN
45,2025-12-01 04:00:00+00:00,874ff9c6-786d-45fc-a90e-48c7ffe03417,41.027636,21.311089,1.998000,100.000000,5.154416,257.905240,940.87067,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
21241,2026-02-27 12:00:00+00:00,2002,41.030221,21.336733,11.513499,45.103836,5.014219,21.037588,953.37820,NaN,NaN
21461,2026-02-28 10:00:00+00:00,2002,41.030221,21.336733,9.413500,65.927530,4.334974,131.633450,954.06915,NaN,NaN
21531,2026-02-28 17:00:00+00:00,2002,41.030221,21.336733,5.563500,80.446945,6.015878,218.927550,951.43207,NaN,NaN
21571,2026-02-28 21:00:00+00:00,2002,41.030221,21.336733,4.363500,76.056366,0.917824,191.309890,952.60974,NaN,NaN


## 5. Save CSV

Missing PM values are written as blank cells. When you load this CSV with pandas later, those blanks become NaN.

In [20]:
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
hourly_nan.to_csv(OUTPUT_CSV, index=False, na_rep="")

print(f"Saved: {OUTPUT_CSV}")

Saved: /mnt/c/Users/RazorVision/Desktop/project-vrnmp/data/streaming/bitola_sensor_weather_features_online_hourly_nan.csv


## 6. Verify that blanks reload as NaN

In [21]:
check = pd.read_csv(OUTPUT_CSV)

print(f"Reloaded rows: {len(check):,}")
print("Missing values after reloading:")
print(check[["pm10", "pm25"]].isna().sum())
print("Missing percent after reloading:")
print((check[["pm10", "pm25"]].isna().mean() * 100).round(2))

check[check[["pm10", "pm25"]].isna().any(axis=1)].head(20)

Reloaded rows: 21,830
Missing values after reloading:
pm10    1482
pm25    1480
dtype: int64
Missing percent after reloading:
pm10    6.79
pm25    6.78
dtype: float64


,timestamp,sensorId,lat,lon,temperature_2m,relative_humidity_2m,wind_speed_10m,wind_direction_10m,surface_pressure,pm10,pm25
5,2025-12-01 00:00:00+00:00,874ff9c6-786d-45fc-a90e-48c7ffe03417,41.027636,21.311089,2.098000,100.000000,6.681856,265.364620,939.69867,NaN,NaN
15,2025-12-01 01:00:00+00:00,874ff9c6-786d-45fc-a90e-48c7ffe03417,41.027636,21.311089,2.448000,100.000000,6.504951,255.579200,940.07050,NaN,NaN
25,2025-12-01 02:00:00+00:00,874ff9c6-786d-45fc-a90e-48c7ffe03417,41.027636,21.311089,2.448000,100.000000,5.297018,260.217650,940.34735,NaN,NaN
35,2025-12-01 03:00:00+00:00,874ff9c6-786d-45fc-a90e-48c7ffe03417,41.027636,21.311089,1.998000,100.000000,4.978554,257.471200,940.50160,NaN,NaN
45,2025-12-01 04:00:00+00:00,874ff9c6-786d-45fc-a90e-48c7ffe03417,41.027636,21.311089,1.998000,100.000000,5.154416,257.905240,940.87067,NaN,NaN
96,2025-12-01 09:00:00+00:00,87f82783-853b-417d-8964-b5cf11e44873,41.035567,21.337528,6.739500,80.329270,5.559640,29.054508,948.93800,NaN,NaN
106,2025-12-01 10:00:00+00:00,87f82783-853b-417d-8964-b5cf11e44873,41.035567,21.337528,8.389500,74.346146,5.876938,40.030197,948.60470,NaN,NaN
116,2025-12-01 11:00:00+00:00,87f82783-853b-417d-8964-b5cf11e44873,41.035567,21.337528,9.539499,70.504480,5.896202,58.736336,948.32990,NaN,NaN
126,2025-12-01 12:00:00+00:00,87f82783-853b-417d-8964-b5cf11e44873,41.035567,21.337528,10.439500,65.691080,8.155807,67.963715,947.99150,NaN,NaN
281,2025-12-02 04:00:00+00:00,2002,41.030221,21.336733,2.463500,89.829056,2.675892,312.273620,948.22577,NaN,17.0
